# Stage 6 — Paired model comparison on the locked EuroSAT test split

This notebook compares the handcrafted statistical baseline, frozen ResNet18 embeddings, and fine-tuned ResNet18 on the **same 4,050 test images**. It uses saved per-image predictions only; it does not train, select, or recalibrate a model. The frozen-model prediction CSV from the original Kaggle run must be present at `results/frozen_resnet18/test_predictions.csv`.

**Question.** How large are the paired differences in classification and probability quality, and how often does one model correct an error made by another?

**Inputs.** The fixed `data/splits/test.csv` manifest and the three `test_predictions.csv` files under `results/`. Predictions are joined by `dataset_index`, then checked for complete coverage, unique indices, matching labels, and valid probabilities. The registered split checksum is `f97c4ec9a27435a932662d5a8b707255`.


In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
from scipy.stats import binomtest
from sklearn.metrics import accuracy_score, f1_score, log_loss

ROOT = Path(r"C:\Users\comviva\Documents\Eurosat_calibrated")
OUTPUT = ROOT / "results" / "paired_model_comparison"
FILES = {
    "handcrafted": ROOT / "results/statistical_baseline/test_predictions.csv",
    "frozen": ROOT / "results/frozen_resnet18/test_predictions.csv",
    "fine_tuned": ROOT / "results/fine_tuned_resnet18/test_predictions.csv",
}
MANIFEST = ROOT / "data/splits/test.csv"
for path in [MANIFEST, *FILES.values()]:
    if not path.is_file():
        raise FileNotFoundError(path)

CLASSES = ["AnnualCrop", "Forest", "HerbaceousVegetation", "Highway",
           "Industrial", "Pasture", "PermanentCrop", "Residential", "River", "SeaLake"]
LABELS = np.arange(10)
PROBABILITY_COLUMNS = [f"probability_{name}" for name in CLASSES]


## Alignment and validation

The three CSV files may have different row orders. Joining each one to the locked manifest by `dataset_index` ensures that paired resamples and correctness comparisons always refer to the same image. Stop if a file is incomplete, duplicates an image, changes a label, or contains invalid probabilities.


In [ ]:
manifest = pd.read_csv(MANIFEST)
if len(manifest) != 4050 or manifest.dataset_index.duplicated().any():
    raise ValueError("Expected the locked 4,050-image test manifest")
manifest = manifest[["dataset_index", "class_index"]].copy()
y_true = manifest.class_index.to_numpy(dtype=int)

def load_predictions(path):
    table = pd.read_csv(path)
    if table.dataset_index.isna().any() or table.dataset_index.duplicated().any():
        raise ValueError(f"Missing or duplicate dataset indices in {path}")
    if set(table.dataset_index) != set(manifest.dataset_index):
        raise ValueError(f"Test image set mismatch in {path}")
    aligned = manifest.merge(table, on="dataset_index", how="left", validate="one_to_one")
    for label_col in ("true_class_index", "class_index_manifest"):
        if label_col in aligned and not np.array_equal(aligned.class_index, aligned[label_col]):
            raise ValueError(f"True-label mismatch in {path}")
    predicted = aligned.predicted_class_index.to_numpy(dtype=int)
    if not np.isin(predicted, LABELS).all():
        raise ValueError(f"Invalid predicted class in {path}")
    probabilities = aligned[PROBABILITY_COLUMNS].to_numpy(dtype=float)
    if (not np.isfinite(probabilities).all() or (probabilities < 0).any()
            or not np.allclose(probabilities.sum(axis=1), 1, atol=1e-4)):
        raise ValueError(f"Invalid probabilities in {path}")
    return predicted, probabilities

loaded = {name: load_predictions(path) for name, path in FILES.items()}
predictions = {name: value[0] for name, value in loaded.items()}
probabilities = {name: value[1] for name, value in loaded.items()}
print(f"Aligned {len(y_true):,} images across all three models")


## Metrics and paired inference

Accuracy and macro-F1 measure classification quality. Multiclass log loss and Brier score measure probability quality; **lower is better** for these two scores. For each model pair, 2,000 bootstrap samples draw test images with replacement and use the **same sampled indices for both models**. A 95% percentile interval is reported for `second − first`. The two-sided exact McNemar test uses only images on which the models disagree about correctness. The bootstrap seed is 42.

These are descriptive comparisons of the three already selected models. The test split is not used to change a model or its temperature.


In [ ]:
def metrics(y, predicted, probs):
    return {
        "accuracy": accuracy_score(y, predicted),
        "macro_f1": f1_score(y, predicted, labels=LABELS, average="macro", zero_division=0),
        "log_loss": log_loss(y, probs, labels=LABELS),
        "multiclass_brier": np.mean(np.sum((probs - np.eye(10)[y]) ** 2, axis=1)),
    }

model_metrics = pd.DataFrame([
    {"model": name, **metrics(y_true, predictions[name], probabilities[name])}
    for name in FILES
])
pairs = [("handcrafted", "frozen"), ("handcrafted", "fine_tuned"),
         ("frozen", "fine_tuned")]
n_bootstrap = 2000
rng = np.random.default_rng(42)
bootstrap_rows, mcnemar_rows = [], []

for first, second in pairs:
    first_metrics = metrics(y_true, predictions[first], probabilities[first])
    second_metrics = metrics(y_true, predictions[second], probabilities[second])
    samples = {name: np.empty(n_bootstrap) for name in first_metrics}
    for iteration in range(n_bootstrap):
        indices = rng.integers(0, len(y_true), size=len(y_true))
        sampled_y = y_true[indices]
        a = metrics(sampled_y, predictions[first][indices], probabilities[first][indices])
        b = metrics(sampled_y, predictions[second][indices], probabilities[second][indices])
        for metric_name in samples:
            samples[metric_name][iteration] = b[metric_name] - a[metric_name]
    for metric_name, differences in samples.items():
        low, high = np.percentile(differences, [2.5, 97.5])
        bootstrap_rows.append({
            "comparison": f"{second} minus {first}", "metric": metric_name,
            "difference": second_metrics[metric_name] - first_metrics[metric_name],
            "ci_low": low, "ci_high": high,
        })
    first_only = int(np.sum((predictions[first] == y_true) & (predictions[second] != y_true)))
    second_only = int(np.sum((predictions[first] != y_true) & (predictions[second] == y_true)))
    discordant = first_only + second_only
    p_value = binomtest(min(first_only, second_only), discordant, 0.5).pvalue if discordant else 1.0
    mcnemar_rows.append({
        "comparison": f"{second} vs {first}", "first_only_correct": first_only,
        "second_only_correct": second_only, "discordant": discordant,
        "mcnemar_exact_p": p_value,
    })

bootstrap_results = pd.DataFrame(bootstrap_rows)
mcnemar_results = pd.DataFrame(mcnemar_rows)
display(model_metrics)
display(bootstrap_results)
display(mcnemar_results)


## Recorded results

The saved CSV outputs from the completed local run give:

| Model | Accuracy | Macro-F1 | Log loss | Multiclass Brier |
|---|---:|---:|---:|---:|
| Handcrafted | 0.747654 | 0.734614 | 0.737415 | 0.360327 |
| Frozen ResNet18 | 0.949383 | 0.947055 | 0.159799 | 0.078636 |
| Fine-tuned ResNet18 | **0.979506** | **0.978540** | **0.074072** | **0.033437** |

Fine-tuning improves accuracy over the frozen representation by **3.012 percentage points** (paired bootstrap 95% interval **2.370 to 3.679 points**) and macro-F1 by **3.149 points** (interval **2.488 to 3.856 points**). Log loss improves by **0.08573** (difference interval **−0.10207 to −0.06801**) and Brier score by **0.04520** (interval **−0.05314 to −0.03732**). All four paired intervals exclude zero.

For the frozen versus fine-tuned comparison, the frozen model alone is correct on **30** images, while the fine-tuned model alone is correct on **152**. The two-sided exact McNemar p-value is **7.70 × 10⁻²¹**. The paired evidence on this fixed test set consistently favors the fine-tuned model.

The fine-tuned model also exceeds the handcrafted baseline by **23.185 accuracy points** (95% interval **21.827 to 24.569 points**) and **24.393 macro-F1 points** (interval **23.092 to 25.764 points**).


## Interpretation and limitations

The progression from handcrafted features to frozen ImageNet features to fine-tuning improves both class predictions and probability scores on the locked EuroSAT RGB test images. The Stage 5 temperature-scaled predictions are deliberately outside this three-model comparison: calibration is a separate analysis of the locked fine-tuned checkpoint, and temperature scaling does not change its class predictions.

The split is image-level, without geographic grouping. Nearby or related Sentinel-2 patches may appear in different partitions, so these results **do not establish generalization to unseen regions**. The paired bootstrap treats test images as independent; spatial dependence could make its intervals too narrow. The three pairwise comparisons and several metrics are exploratory and are not adjusted for multiplicity. The test set has been examined in earlier project stages; it should remain locked against further model or calibration tuning.

**Next stage:** inspect representative mistakes and disagreement cases, especially PermanentCrop, River, and Highway, then prepare the final research report and portfolio presentation.


In [ ]:
OUTPUT.mkdir(parents=True, exist_ok=True)
model_metrics.to_csv(OUTPUT / "model_metrics.csv", index=False)
bootstrap_results.to_csv(OUTPUT / "paired_bootstrap.csv", index=False)
mcnemar_results.to_csv(OUTPUT / "mcnemar_exact.csv", index=False)
method = {
    "bootstrap_resamples": n_bootstrap,
    "random_seed": 42,
    "bootstrap_unit": "test image",
    "confidence_interval": "95% percentile interval",
    "difference_order": "second model minus first model",
    "mcnemar_test": "two-sided exact binomial",
    "test_samples": len(y_true),
    "split_checksum": "f97c4ec9a27435a932662d5a8b707255",
}
(OUTPUT / "method.json").write_text(json.dumps(method, indent=2), encoding="utf-8")
print("Saved outputs to:", OUTPUT)
